## Notebook 11 — RF Feature Importance Exploration
**Project:** Machine Learning for High Performance Optical Sorting
**Author:** Mohamed Tawfeek
**Description:** Detailed exploration of Random Forest feature importances broken down by H, S, V, and LBP channel groups


In [1]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import pickle
import os

In [2]:
RESULTS_DIR = "results"
RF_MODEL_PATH = os.path.join('..', RESULTS_DIR, "rf_final_model.pkl")
FIGURES_DIR  = os.path.join('..', RESULTS_DIR, "figures")

In [3]:
# Loads the trained RF model and extracts mean impurity decrease (Gini importance) for each of the 122 features

with open(RF_MODEL_PATH, "rb") as f:
    rf = pickle.load(f)
 
importances = rf.feature_importances_ 
n_features  = len(importances)          

In [4]:
groups = {
    "H (Hue)":       (0,  32),
    "S (Saturation)":(32, 64),
    "V (Value)":     (64, 96),
    "LBP (Texture)": (96, 122),
}
 
colors = {
    "H (Hue)":        "#2196F3",   # blue
    "S (Saturation)": "#4CAF50",   # green
    "V (Value)":      "#FF9800",   # orange
    "LBP (Texture)":  "#9C27B0",   # purple
}

In [5]:
# Sums feature importances within each channel group and prints the group-level proportions

group_totals = {name: importances[lo:hi].sum() for name, (lo, hi) in groups.items()}
 
print("=" * 55)
print(f"{'Group':<20} {'Total Imp':>10} {'Proportion':>12}")
print("=" * 55)
for name, total in group_totals.items():
    print(f"{name:<20} {total:>10.4f} {total*100:>11.1f}%")
print("=" * 55)
print(f"{'TOTAL':<20} {sum(group_totals.values()):>10.4f}")

Group                 Total Imp   Proportion
H (Hue)                  0.2279        22.8%
S (Saturation)           0.2469        24.7%
V (Value)                0.2545        25.5%
LBP (Texture)            0.2707        27.1%
TOTAL                    1.0000


In [6]:
# Bar chart of all 122 feature importances coloured by channel group, with the zero-importance hue region annotated

fig, ax = plt.subplots(figsize=(14, 4.5))
 
bar_colors = []
for i in range(n_features):
    if i < 32:   bar_colors.append(colors["H (Hue)"])
    elif i < 64: bar_colors.append(colors["S (Saturation)"])
    elif i < 96: bar_colors.append(colors["V (Value)"])
    else:        bar_colors.append(colors["LBP (Texture)"])
 
ax.bar(np.arange(n_features), importances, color=bar_colors, width=1.0, linewidth=0)
 
for boundary in [32, 64, 96]:
    ax.axvline(boundary - 0.5, color="black", linewidth=1.2, linestyle="--", alpha=0.5)
 
ax.axvspan(22.5, 31.5, alpha=0.15, color="red")
ax.annotate(
    "Zero importance:\nH bins 23–31\n(cyan/blue/violet hues\nabsent in waste data)",
    xy=(27, 0.00015),
    xytext=(42, 0.014),
    fontsize=7.5,
    color="red",
    arrowprops=dict(arrowstyle="->", color="red", lw=1.2),
)
 
patches = [mpatches.Patch(color=colors[n], label=n) for n in groups]
ax.legend(handles=patches, fontsize=8.5, loc="upper right")
 
ax.set_xlabel("Feature Index", fontsize=10)
ax.set_ylabel("Mean Impurity Decrease", fontsize=10)
ax.set_title(
    "Random Forest Feature Importances — Coloured by H / S / V / LBP Group",
    fontsize=11
)
ax.set_xlim(-1, 122)
ax.set_ylim(bottom=0)
 
for name, (lo, hi) in groups.items():
    ax.text((lo + hi) / 2, ax.get_ylim()[1] * 0.97,
            name.split()[0], ha="center", va="top",
            fontsize=9, fontweight="bold", color=colors[name])
 
plt.tight_layout()
out1 = os.path.join(FIGURES_DIR, "rf_feature_importances_grouped.png")
plt.savefig(out1, dpi=150, bbox_inches="tight")
plt.close()
print(f"\nSaved: {out1}")


Saved: ..\results\figures\rf_feature_importances_grouped.png


In [7]:
# Four-panel figure with a separate importance bar chart per channel group, each annotated with its total importance share

fig, axes = plt.subplots(1, 4, figsize=(16, 4),
                         gridspec_kw={"width_ratios": [32, 32, 32, 26]})
 
for ax, (name, (lo, hi)) in zip(axes, groups.items()):
    local_x   = np.arange(hi - lo)
    local_imp = importances[lo:hi]
    ax.bar(local_x, local_imp, color=colors[name], width=0.85)
    ax.set_title(name, fontsize=9, fontweight="bold", color=colors[name])
    ax.set_xlabel(f"Bin (within {name.split()[0]})", fontsize=8)
    if ax is axes[0]:
        ax.set_ylabel("Mean Impurity Decrease", fontsize=8)
    total = local_imp.sum()
    ax.text(0.97, 0.96, f"Total: {total:.4f}\n({total*100:.1f}%)",
            transform=ax.transAxes, ha="right", va="top", fontsize=7.5,
            bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.8))
    ax.set_ylim(bottom=0)
 
    if name == "H (Hue)":
        ax.axvspan(22.5, 31.5, alpha=0.15, color="red")
        ax.text(27, ax.get_ylim()[1] * 0.5,
                "Zero\n(cyan–\nviolet)",
                ha="center", fontsize=6.5, color="red")
 
plt.suptitle("Feature Importances by Channel Group — RF on TrashNet HSV+LBP Features",
             fontsize=10, y=1.02)
plt.tight_layout()
out2 = os.path.join(FIGURES_DIR, "rf_feature_importances_by_group.png")
plt.savefig(out2, dpi=150, bbox_inches="tight")
plt.close()
print(f"Saved: {out2}")

Saved: ..\results\figures\rf_feature_importances_by_group.png


In [8]:
# Single bar chart summarising total importance across the four channel groups

fig, ax = plt.subplots(figsize=(6, 4))
names  = list(group_totals.keys())
totals = list(group_totals.values())
bars = ax.bar(names, totals,
              color=[colors[n] for n in names],
              width=0.5, edgecolor="white")
 
for bar, val in zip(bars, totals):
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.003,
            f"{val:.4f}\n({val*100:.1f}%)",
            ha="center", va="bottom", fontsize=9)
 
ax.set_ylabel("Total Mean Impurity Decrease", fontsize=10)
ax.set_title("Feature Group Importance — RF Summary", fontsize=11)
ax.set_ylim(0, max(totals) * 1.3)
ax.tick_params(axis="x", labelsize=9)
plt.tight_layout()
out3 = os.path.join(FIGURES_DIR, "rf_feature_importances_group_summary.png")
plt.savefig(out3, dpi=150, bbox_inches="tight")
plt.close()
print(f"Saved: {out3}")
 

Saved: ..\results\figures\rf_feature_importances_group_summary.png
